# Vantara 3D: train a Gaussian splat on a Colab GPU (gsplat fallback)

GPU fallback for Phase 4 when Brush is too slow on the local machine. Trains
[gsplat](https://github.com/nerfstudio-project/gsplat) 1.5.3 (`simple_trainer.py`)
for 30k steps on a zone dataset and exports a PLY.

**Input:** the zip made locally by

```bash
uv run train-splat <zone> --prepare-only --zip
# -> pipeline/work/<zone>/splat/<zone>-dataset.zip  (images/, masks/, sparse/0/)
```

`masks/` (burnt-in subtitles, logos) are honoured: masked pixels are excluded from the loss.

**Runtime:** Runtime > Change runtime type > T4 GPU (or better). About 30-60 min
for about 200 images on a T4. The first run also compiles gsplat's CUDA kernels (5-10 min).

**Privacy:** this uploads the dataset to Google. Only use footage you have the rights to.

In [ ]:
# @title Settings
ZONE = "elephant-care-centre"  # @param {type:"string"}
# Path to the dataset zip in Google Drive; leave empty to upload from this computer instead.
DATASET_ZIP = "/content/drive/MyDrive/vantara3d/elephant-care-centre-dataset.zip"  # @param {type:"string"}
STEPS = 30000  # @param {type:"integer"}
# Downscale factor for training images (2 = 2560px frames train at 1280px).
DATA_FACTOR = 2  # @param {type:"integer"}
# "default" = classic densification, "mcmc" = fixed splat budget (good for size caps)
STRATEGY = "default"  # @param ["default", "mcmc"]
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
DOWNLOAD_PLY = False  # @param {type:"boolean"}

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title Get the dataset
import os, shutil, zipfile
from google.colab import drive, files

if DATASET_ZIP.startswith("/content/drive") or SAVE_TO_DRIVE:
    drive.mount("/content/drive")
if DATASET_ZIP:
    zip_path = DATASET_ZIP
else:
    uploaded = files.upload()
    zip_path = "/content/" + next(iter(uploaded))

DATA_DIR = f"/content/data/{ZONE}"
shutil.rmtree(DATA_DIR, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATA_DIR)
n_img = len(os.listdir(f"{DATA_DIR}/images"))
n_mask = len(os.listdir(f"{DATA_DIR}/masks")) if os.path.isdir(f"{DATA_DIR}/masks") else 0
assert os.path.isfile(f"{DATA_DIR}/sparse/0/cameras.bin"), "zip must contain sparse/0/cameras.bin"
print(f"{n_img} images, {n_mask} masks in {DATA_DIR}")

In [ ]:
# @title Install gsplat 1.5.3 (pinned)
!git clone -q --depth 1 --branch v1.5.3 https://github.com/nerfstudio-project/gsplat.git /content/gsplat || true
!pip install -q ninja gsplat==1.5.3
!pip install -q -r /content/gsplat/examples/requirements.txt
# Training runs in a subprocess, so the example's numpy<2 pin needs no runtime restart
# (pip warns about Colab packages that want numpy 2; training doesn't use them).

In [ ]:
%%writefile /content/run_gsplat.py
# Runs gsplat's examples/simple_trainer.py with per-image masks from <data_dir>/masks.
# The stock trainer only zeroes the *render* inside a mask; we also zero the
# ground-truth pixels there, so masked areas (subtitles, logos) carry no loss.
import os
import runpy
import sys

import types

import cv2
import torch

EXAMPLES = "/content/gsplat/examples"
sys.path.insert(0, EXAMPLES)
os.chdir(EXAMPLES)
# examples/datasets has no __init__.py, so Colab's preinstalled Hugging Face
# `datasets` package would shadow it whatever the path order: register it by hand.
_pkg = types.ModuleType("datasets")
_pkg.__path__ = [os.path.join(EXAMPLES, "datasets")]
sys.modules["datasets"] = _pkg
# gsplat's examples pin numpy<2 (its COLMAP reader breaks on numpy 2), but
# TensorBoard would import Colab's TensorFlow, which needs numpy 2. Hide TF so
# TensorBoard falls back to its built-in stub.
sys.modules["tensorflow"] = None
from datasets import colmap as colmap_ds  # noqa: E402

MASK_DIR = os.environ.get("MASK_DIR", "")
_orig_getitem = colmap_ds.Dataset.__getitem__


def _getitem(self, item):
    data = _orig_getitem(self, item)
    name = os.path.splitext(os.path.basename(self.parser.image_names[self.indices[item]]))[0]
    path = os.path.join(MASK_DIR, name + ".png")
    if MASK_DIR and os.path.isfile(path):
        h, w = data["image"].shape[:2]
        keep = cv2.resize(cv2.imread(path, cv2.IMREAD_GRAYSCALE), (w, h), interpolation=cv2.INTER_NEAREST) > 127
        keep = torch.from_numpy(keep)
        data["mask"] = keep & data["mask"] if "mask" in data else keep
        data["image"][~keep] = 0
    return data


colmap_ds.Dataset.__getitem__ = _getitem
sys.argv[0] = os.path.join(EXAMPLES, "simple_trainer.py")
runpy.run_path(sys.argv[0], run_name="__main__")

In [ ]:
# @title Train
import os
RESULT_DIR = f"/content/results/{ZONE}"
mask_dir = f"{DATA_DIR}/masks" if os.path.isdir(f"{DATA_DIR}/masks") else ""
!MASK_DIR="{mask_dir}" python /content/run_gsplat.py {STRATEGY}     --data_dir "{DATA_DIR}" --data_factor {DATA_FACTOR} --result_dir "{RESULT_DIR}"     --max_steps {STEPS} --save_ply --ply_steps 7000 {STEPS}     --test_every 10 --disable_viewer

In [ ]:
# @title Collect the PLY
import glob, os, re, shutil
plys = sorted(glob.glob(f"{RESULT_DIR}/ply/*.ply"), key=lambda p: int(re.findall(r"(\d+)", os.path.basename(p))[-1]))
assert plys, "no PLY exported - check the training log above"
final = f"/content/{ZONE}.ply"
shutil.copy(plys[-1], final)
print(f"{final}: {os.path.getsize(final) / 1e6:.1f} MB")
!cat {RESULT_DIR}/stats/val_step*.json 2>/dev/null | tail -c 400

if SAVE_TO_DRIVE:
    out = f"/content/drive/MyDrive/vantara3d/{ZONE}.ply"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    shutil.copy(final, out)
    print("saved to", out)
if DOWNLOAD_PLY:
    from google.colab import files
    files.download(final)

Put the PLY at `pipeline/work/<zone>/splat/<zone>.ply` locally; Phase 5 (cleanup,
alignment, SPZ) picks it up from there.